# SP-GaRT with Velocity Residual Training Notebook

## What This Notebook Does Differently

This notebook trains the same five models (M1, M2, M3a, M4, M5) but with one change:
the models predict **velocity residuals** instead of absolute joint positions.

### What is a Velocity Residual?

**Original approach (absolute prediction):**
The model sees 10 frames of skeleton and directly predicts where each joint will be for the next 25 frames. These are absolute 3D coordinates in metres.

**Velocity residual approach:**
Instead of predicting where joints will be, the model predicts how much each joint will move from its last observed position.

```
Absolute:  predict joint_position_at_frame_11 = (0.42, 0.15, 0.83)
Residual:  predict delta_from_frame_10 = (+0.02, -0.01, +0.03)
           then add: frame_10_position + delta = final prediction
```

### Why Residuals Are Easier to Predict

The changes (deltas) between consecutive frames are tiny numbers close to zero.
A person moving normally changes their joint positions by roughly 5-30mm per frame.
This is a much smaller and more predictable target than absolute coordinates
which span the range -1.5m to +2.0m.

Think of it like giving directions:
- Absolute: 'Go to coordinate (51.5074, -0.1278)' requires knowing the global map
- Residual: 'Walk 500m north from where you are' requires only knowing the direction

The model only needs to learn motion patterns, not absolute body positions.
This is why papers using residual prediction consistently achieve lower MPJPE.

### Novel Contributions

1. Graph structure with anatomical bias (M2)
2. Heuristic spatial pruning (M3a)
3. Gravity-consistency loss (M4/M5)

### Checkpoint Locations

- SP_GaRT_VelRes/checkpoints/
- SP_GaRT_VelRes/runs/         

## 01. Environment Setup

### 1.1 Verify GPU

In [1]:
import torch

print(f'PyTorch : {torch.__version__}')
print(f'GPU     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device  : {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: No GPU found. Training will be very slow.')

PyTorch : 2.11.0+cu128
GPU     : True
Device  : Tesla T4


### 1.2 Mount Drive and Set Paths

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── Original project (DO NOT WRITE HERE) ─────────────────────
DRIVE_ORIGINAL = '/content/drive/MyDrive/L4_Research_Resources/SP_GaRT'

# ── Velocity residual version (all new outputs go here) ──────
DRIVE_PROJECT = '/content/drive/MyDrive/L4_Research_Resources/SP_GaRT_VelRes'
SAVE_DIR      = f'{DRIVE_PROJECT}/checkpoints'
LOG_DIR       = f'{DRIVE_PROJECT}/runs'

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(LOG_DIR,  exist_ok=True)
os.makedirs(f'{DRIVE_ORIGINAL}/data', exist_ok=True)

print(f'Original project  : {DRIVE_ORIGINAL}  (read-only — your trained models)')
print(f'VelRes project    : {DRIVE_PROJECT}   (new checkpoints go here)')
print(f'Checkpoints dir   : {SAVE_DIR}')
print(f'Logs dir          : {LOG_DIR}')

Mounted at /content/drive
Original project  : /content/drive/MyDrive/L4_Research_Resources/SP_GaRT  (read-only — your trained models)
VelRes project    : /content/drive/MyDrive/L4_Research_Resources/SP_GaRT_VelRes   (new checkpoints go here)
Checkpoints dir   : /content/drive/MyDrive/L4_Research_Resources/SP_GaRT_VelRes/checkpoints
Logs dir          : /content/drive/MyDrive/L4_Research_Resources/SP_GaRT_VelRes/runs


### 1.3 Clone Repository

In [3]:
if not os.path.exists('/content/SP_GaRT'):
    !git clone https://github.com/GayuniBas2001/SP-GaRT_Spatially_Pruned_Graph_Transformer.git /content/SP_GaRT
    print('Cloned.')
else:
    !cd /content/SP_GaRT && git pull
    print('Updated.')

%cd /content/SP_GaRT

import sys
sys.path.insert(0, '/content/SP_GaRT')
print('Path set.')

Cloning into '/content/SP_GaRT'...
remote: Enumerating objects: 87, done.
remote: Counting objects: 100% (87/87), done.
remote: Compressing objects: 100% (64/64), done.
remote: Total 87 (delta 27), reused 68 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (87/87), 369.85 KiB | 16.81 MiB/s, done.
Resolving deltas: 100% (27/27), done.
Cloned.
/content/SP_GaRT
Path set.


### 1.4 Copy Data and Install Dependencies

In [4]:
DATA_LOCAL = '/content/SP_GaRT/data/data_3d_h36m.npz'
DATA_DRIVE = f'{DRIVE_ORIGINAL}/data/data_3d_h36m.npz'

if not os.path.exists(DATA_LOCAL):
    print('Copying data from Drive...')
    !cp "{DATA_DRIVE}" "{DATA_LOCAL}"
    print('Done.')
else:
    print('Data already present.')

print(f'File size: {os.path.getsize(DATA_LOCAL)/1e6:.1f}MB')

Copying data from Drive...
Done.
File size: 153.2MB


In [5]:
!pip install tensorboard -q
print('tensorboard ready.')

tensorboard ready.


### 1.5 Load Dataset

In [6]:
from data.h36m_dataset import build_dataloaders
from utils.metrics import (
    mpjpe_at_horizons, gravity_violation_rate, bone_length_error
)
from utils.trainer import evaluate_model, print_results, measure_inference_time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train_loader, test_loader = build_dataloaders(
    DATA_LOCAL, batch_size=32,
    t_obs=10, t_pred=25,
    train_stride=5, test_stride=1
)

batch = next(iter(train_loader))
print(f'Observed : {batch["observed"].shape}')
print(f'Future   : {batch["future"].shape}')
print(f'Device   : {device}')

[H36MDataset] 38045 windows | subjects=['S1', 'S5', 'S6', 'S7', 'S8'] | t_obs=10 t_pred=25 | stride=5 | 25Hz
[H36MDataset] 65927 windows | subjects=['S9', 'S11'] | t_obs=10 t_pred=25 | stride=1 | 25Hz
Observed : torch.Size([32, 10, 17, 3])
Future   : torch.Size([32, 25, 17, 3])
Device   : cuda


## 02. Velocity Residual Wrappers

The ONLY new code compared to the original SP-GaRT training notebook.

### How the Wrapper Works

We wrap each model with a thin layer that handles the conversion:

**During training (forward pass):**
1. Take the last observed frame as the 'anchor' position
2. Compute target residuals = future_absolute - last_observed_frame
   (these are the tiny deltas the model needs to learn)
3. Run the model : it predicts residuals (small numbers near zero)
4. Reconstruct absolute positions = last_observed + predicted_residuals
5. Compute loss on the reconstructed absolute positions
   (so MPJPE is still in mm and comparable to original results)
6. For gravity loss: apply on reconstructed absolute positions

**During evaluation:**
Same reconstruction: predicted_absolute = last_observed + predicted_residuals
Then compute MPJPE, GVR, BLE exactly as before on absolute coordinates.

### Why We Use a Wrapper, Not a New Model

The models themselves (M1, M2, M3a, M4, M5) are completely unchanged.
The wrapper is just a conversion layer that sits between the data and the model.
This means:
- All graph structure, attention mechanisms, and pruning are identical
- The gravity loss still checks absolute positions
- We can directly compare with original results
- No risk of breaking the model architecture

In [7]:
import torch
import torch.nn as nn
from utils.metrics import gravity_consistency_loss

# ─────────────────────────────────────────────────────────────
# VELOCITY RESIDUAL WRAPPER
# ─────────────────────────────────────────────────────────────
class VelocityResidualWrapper(nn.Module):
    """
    Wraps any SP-GaRT model to use velocity residual prediction.

    The underlying model is UNCHANGED. This wrapper:
    1. Subtracts the last observed frame from the future sequence
       to create residual targets (small deltas near zero)
    2. The model learns to predict these small deltas
    3. Adds the last observed frame back to get absolute predictions

    Why this helps:
    - Residuals are small numbers (5-30mm per frame)
    - Absolute positions are large numbers (-1500mm to +2000mm)
    - Small numbers are much easier to predict accurately
    - Models converge faster and to lower error

    Args:
        base_model: any of VanillaTransformer, DenseGraphTransformer,
                    PrunedGraphTransformer etc.
    """
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model

    def forward(self, observed):
        """
        Args:
            observed: (B, T_obs, J, 3) — observed skeleton sequence
        Returns:
            predicted_absolute: (B, T_pred, J, 3) — absolute predictions
                                (NOT residuals — already reconstructed)
        """
        # The anchor is the last observed frame
        # Shape: (B, 1, J, 3) — kept as 4D for broadcasting
        last_frame = observed[:, -1:, :, :]

        # The base model predicts residuals (deltas)
        # These are small values like (+0.02, -0.01, +0.03)
        # rather than large absolute coords like (0.42, 0.15, 0.83)
        predicted_residuals = self.base_model(observed)
        # Shape: (B, T_pred, J, 3)

        # Reconstruct absolute positions by adding last observed frame
        # last_frame broadcasts across all 25 prediction frames
        predicted_absolute = last_frame + predicted_residuals
        # Shape: (B, T_pred, J, 3)

        return predicted_absolute

    def count_parameters(self):
        return sum(
            p.numel() for p in self.parameters()
            if p.requires_grad
        )


# ─────────────────────────────────────────────────────────────
# VELOCITY RESIDUAL TRAINING FUNCTION (standard loss)
# Used for M1, M2, M3a
# ─────────────────────────────────────────────────────────────
def train_model_velres(model_wrapper, train_loader, test_loader,
                        n_epochs, lr, device,
                        experiment_name,
                        eval_every=5,
                        save_dir='checkpoints',
                        log_dir='runs',
                        resume=True):
    """
    Training loop for velocity residual models (M1, M2, M3a).

    Key difference from original train_model:
    - The loss is computed on RESIDUALS not absolute positions
    - target_residuals = future_absolute - last_observed_frame
    - predicted_residuals = model(observed)  [before wrapper adds back]
    - loss = MSE(predicted_residuals, target_residuals)

    The evaluation still uses absolute positions (MPJPE in mm)
    because the wrapper reconstructs them automatically.

    Why train on residuals but evaluate on absolute?
    Training on small numbers converges faster.
    Evaluating on absolute mm makes results comparable to literature.
    """
    import os
    from torch.utils.tensorboard import SummaryWriter

    os.makedirs(save_dir, exist_ok=True)
    os.makedirs(f'{log_dir}/{experiment_name}', exist_ok=True)

    writer    = SummaryWriter(f'{log_dir}/{experiment_name}')
    optimizer = torch.optim.Adam(
        model_wrapper.parameters(), lr=lr
    )
    scheduler = torch.optim.lr_scheduler.MultiStepLR(
        optimizer, milestones=[30, 60, 80], gamma=0.5
    )
    loss_fn = nn.MSELoss()

    best_mpjpe  = float('inf')
    start_epoch = 1
    latest_path = f'{save_dir}/{experiment_name}_latest.pth'
    best_path   = f'{save_dir}/{experiment_name}_best.pth'

    if resume and os.path.exists(latest_path):
        print(f'Resuming: {latest_path}')
        ckpt = torch.load(latest_path, map_location=device)
        model_wrapper.load_state_dict(ckpt['model_state'])
        optimizer.load_state_dict(ckpt['optimizer_state'])
        scheduler.load_state_dict(ckpt['scheduler_state'])
        start_epoch = ckpt['epoch'] + 1
        best_mpjpe  = ckpt.get('best_mpjpe', float('inf'))
        print(f'  Resumed epoch {start_epoch} | '
              f'Best: {best_mpjpe:.1f}mm')
    else:
        print(f'Starting: {experiment_name}')

    for epoch in range(start_epoch, n_epochs + 1):
        model_wrapper.train()
        total_loss = 0.0
        n_batches  = 0

        for batch in train_loader:
            obs = batch['observed'].to(device)  # (B, T_obs, J, 3)
            fut = batch['future'].to(device)    # (B, T_pred, J, 3)

            # ── Compute residual targets ──────────────────────────
            # What are residuals? The CHANGE from the last observed
            # frame to each future frame.
            #
            # Example:
            #   last_frame joint_0 = (0.40, 0.15, 0.83)
            #   future_frame_1 joint_0 = (0.42, 0.14, 0.84)
            #   residual = (0.02, -0.01, 0.01)  ← tiny!
            #
            # The model learns to predict these tiny deltas
            # instead of large absolute coordinates.
            last_frame = obs[:, -1:, :, :]      # (B, 1, J, 3)
            target_residuals = fut - last_frame  # (B, T_pred, J, 3)

            optimizer.zero_grad()

            # The base model inside the wrapper predicts residuals
            # The wrapper adds last_frame back to get absolute
            # We need the RAW residuals for the loss, so we call
            # base_model directly here (before the +last_frame step)
            predicted_residuals = model_wrapper.base_model(obs)

            # Loss is between predicted residuals and target residuals
            # Both are small numbers — much easier to minimise
            loss = loss_fn(predicted_residuals, target_residuals)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model_wrapper.parameters(), max_norm=1.0
            )
            optimizer.step()

            total_loss += loss.item()
            n_batches  += 1

        avg_loss = total_loss / n_batches
        scheduler.step()

        writer.add_scalar('Loss/train', avg_loss, epoch)
        writer.add_scalar(
            'LR', optimizer.param_groups[0]['lr'], epoch
        )

        # Save latest checkpoint every epoch
        torch.save({
            'epoch':           epoch,
            'model_state':     model_wrapper.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'scheduler_state': scheduler.state_dict(),
            'best_mpjpe':      best_mpjpe,
        }, latest_path)

        if epoch % eval_every == 0:
            # evaluate_model calls model_wrapper.forward()
            # which reconstructs absolute positions automatically
            # so MPJPE is still in mm and directly comparable
            results   = evaluate_model(
                model_wrapper, test_loader, device, n_batches=50
            )
            mpjpe_560 = results['mpjpe'][560]
            gvr       = results['gvr']

            writer.add_scalar('MPJPE/560ms', mpjpe_560, epoch)
            if gvr is not None:
                writer.add_scalar('GVR', gvr, epoch)

            gvr_str = f'{gvr:.4f}' if gvr is not None else 'N/A'
            print(f'Epoch {epoch:>3}/{n_epochs} | '
                  f'loss: {avg_loss:.5f} | '
                  f'MPJPE@560ms: {mpjpe_560:.1f}mm | '
                  f'GVR: {gvr_str}')

            if mpjpe_560 < best_mpjpe:
                best_mpjpe = mpjpe_560
                torch.save({
                    'epoch':       epoch,
                    'model_state': model_wrapper.state_dict(),
                    'results':     results,
                    'best_mpjpe':  best_mpjpe,
                }, best_path)
                print(f'  ✓ Best saved: MPJPE@560ms={best_mpjpe:.1f}mm')

        elif epoch % 5 == 0:
            print(f'Epoch {epoch:>3}/{n_epochs} | '
                  f'loss: {avg_loss:.5f}')

    writer.close()
    print(f'\nDone. Best MPJPE@560ms: {best_mpjpe:.1f}mm')
    print(f'Best checkpoint: {best_path}')
    return model_wrapper


# ─────────────────────────────────────────────────────────────
# VELOCITY RESIDUAL + GRAVITY LOSS TRAINING FUNCTION
# Used for M4 and M5
# ─────────────────────────────────────────────────────────────
class VelResGravityLoss(nn.Module):
    """
    Combined loss for velocity residual models with gravity constraint.

    L_total = L_recon_residual + lambda * L_gravity_absolute

    L_recon_residual:
        MSE between predicted residuals and target residuals.
        Residuals are small deltas, making reconstruction easier.

    L_gravity_absolute:
        The gravity loss always checks ABSOLUTE positions.
        Whether a person is balanced depends on where their
        joints actually are, not on how much they moved.
        So we reconstruct absolute positions first, then check.

    This means the gravity contribution is completely unchanged
    from the original M4/M5 — same physics, same threshold,
    same BoS definition. Only the accuracy loss changes.
    """
    def __init__(self, lambda_gravity=0.01):
        super().__init__()
        self.lambda_gravity = lambda_gravity
        self.mse = nn.MSELoss()

    def forward(self, predicted_residuals, future_absolute,
                observed, last_frame):
        """
        Args:
            predicted_residuals: (B, T_pred, J, 3) — model output (raw)
            future_absolute:     (B, T_pred, J, 3) — ground truth absolute
            observed:            (B, T_obs,  J, 3) — input
            last_frame:          (B, 1,      J, 3) — anchor for reconstruction
        """
        # Accuracy loss on residuals (small numbers)
        target_residuals = future_absolute - last_frame
        l_recon = self.mse(predicted_residuals, target_residuals)

        # Reconstruct absolute for gravity check
        # Physics must be checked on real world coordinates
        predicted_absolute = last_frame + predicted_residuals
        l_gravity = gravity_consistency_loss(
            predicted_absolute, observed
        )

        total = l_recon + self.lambda_gravity * l_gravity
        return total, l_recon, l_gravity


def train_model_velres_gravity(model_wrapper, train_loader,
                                test_loader, n_epochs, lr,
                                device, experiment_name,
                                lambda_gravity=0.01,
                                eval_every=5,
                                save_dir='checkpoints',
                                log_dir='runs',
                                resume=True):
    """
    Training loop for velocity residual models WITH gravity loss.
    Used for M4 (dense graph + gravity) and M5 (pruned + gravity).

    The gravity loss is identical to the original:
    - Checks if CoM proxy (root joint) is within ankle BoS
    - Applied only to standing frames (root Z > 0.70m)
    - Lambda = 0.01 keeps gravity as a small regulariser
    - Zero additional inference overhead
    """
    import os
    from torch.utils.tensorboard import SummaryWriter

    os.makedirs(save_dir, exist_ok=True)
    os.makedirs(f'{log_dir}/{experiment_name}', exist_ok=True)

    writer    = SummaryWriter(f'{log_dir}/{experiment_name}')
    optimizer = torch.optim.Adam(
        model_wrapper.parameters(), lr=lr
    )
    scheduler = torch.optim.lr_scheduler.MultiStepLR(
        optimizer, milestones=[30, 60, 80], gamma=0.5
    )
    loss_fn = VelResGravityLoss(lambda_gravity=lambda_gravity)

    best_mpjpe  = float('inf')
    start_epoch = 1
    latest_path = f'{save_dir}/{experiment_name}_latest.pth'
    best_path   = f'{save_dir}/{experiment_name}_best.pth'

    if resume and os.path.exists(latest_path):
        print(f'Resuming: {latest_path}')
        ckpt = torch.load(latest_path, map_location=device)
        model_wrapper.load_state_dict(ckpt['model_state'])
        optimizer.load_state_dict(ckpt['optimizer_state'])
        scheduler.load_state_dict(ckpt['scheduler_state'])
        start_epoch = ckpt['epoch'] + 1
        best_mpjpe  = ckpt.get('best_mpjpe', float('inf'))
        print(f'  Resumed epoch {start_epoch} | '
              f'Best: {best_mpjpe:.1f}mm')
    else:
        print(f'Starting: {experiment_name}')

    for epoch in range(start_epoch, n_epochs + 1):
        model_wrapper.train()
        total_loss  = 0.0
        total_recon = 0.0
        total_grav  = 0.0
        n_batches   = 0

        for batch in train_loader:
            obs = batch['observed'].to(device)
            fut = batch['future'].to(device)

            last_frame = obs[:, -1:, :, :]  # anchor

            optimizer.zero_grad()

            # Raw residual prediction from base model
            predicted_residuals = model_wrapper.base_model(obs)

            # Loss: residual accuracy + gravity on absolute
            loss, l_recon, l_grav = loss_fn(
                predicted_residuals, fut, obs, last_frame
            )
            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model_wrapper.parameters(), max_norm=1.0
            )
            optimizer.step()

            total_loss  += loss.item()
            total_recon += l_recon.item()
            total_grav  += l_grav.item()
            n_batches   += 1

        avg_loss  = total_loss  / n_batches
        avg_recon = total_recon / n_batches
        avg_grav  = total_grav  / n_batches
        scheduler.step()

        writer.add_scalar('Loss/total',   avg_loss,  epoch)
        writer.add_scalar('Loss/recon',   avg_recon, epoch)
        writer.add_scalar('Loss/gravity', avg_grav,  epoch)
        writer.add_scalar(
            'LR', optimizer.param_groups[0]['lr'], epoch
        )

        torch.save({
            'epoch':           epoch,
            'model_state':     model_wrapper.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'scheduler_state': scheduler.state_dict(),
            'best_mpjpe':      best_mpjpe,
        }, latest_path)

        if epoch % eval_every == 0:
            results   = evaluate_model(
                model_wrapper, test_loader, device, n_batches=50
            )
            mpjpe_560 = results['mpjpe'][560]
            gvr       = results['gvr']

            writer.add_scalar('MPJPE/560ms', mpjpe_560, epoch)
            if gvr is not None:
                writer.add_scalar('GVR', gvr, epoch)

            gvr_str = f'{gvr:.4f}' if gvr is not None else 'N/A'
            print(f'Epoch {epoch:>3}/{n_epochs} | '
                  f'loss: {avg_loss:.5f} '
                  f'(recon: {avg_recon:.5f}, '
                  f'grav: {avg_grav:.5f}) | '
                  f'MPJPE@560ms: {mpjpe_560:.1f}mm | '
                  f'GVR: {gvr_str}')

            if mpjpe_560 < best_mpjpe:
                best_mpjpe = mpjpe_560
                torch.save({
                    'epoch':       epoch,
                    'model_state': model_wrapper.state_dict(),
                    'results':     results,
                    'best_mpjpe':  best_mpjpe,
                }, best_path)
                print(f'  ✓ Best saved: MPJPE@560ms={best_mpjpe:.1f}mm')

        elif epoch % 5 == 0:
            print(f'Epoch {epoch:>3}/{n_epochs} | '
                  f'loss: {avg_loss:.5f} '
                  f'(recon: {avg_recon:.5f}, '
                  f'grav: {avg_grav:.5f})')

    writer.close()
    print(f'\nDone. Best MPJPE@560ms: {best_mpjpe:.1f}mm')
    print(f'Best checkpoint: {best_path}')
    return model_wrapper


print('✓ VelocityResidualWrapper defined')
print('✓ train_model_velres defined (for M1, M2, M3a)')
print('✓ VelResGravityLoss defined')
print('✓ train_model_velres_gravity defined (for M4, M5)')

✓ VelocityResidualWrapper defined
✓ train_model_velres defined (for M1, M2, M3a)
✓ VelResGravityLoss defined
✓ train_model_velres_gravity defined (for M4, M5)


### Verify the Wrapper Works

Before training anything, confirm the wrapper runs correctly
and produces the right output shapes.

In [8]:
# Quick sanity check of the wrapper
from models.vanilla_transformer import VanillaTransformer

base_m1 = VanillaTransformer(
    J=17, D=3, d_model=256, n_heads=4,
    n_enc_layers=2, n_dec_layers=2,
    d_ff=512, dropout=0.1, t_obs=10, t_pred=25
).to(device)

wrapper_test = VelocityResidualWrapper(base_m1).to(device)

sample   = next(iter(test_loader))
obs_test = sample['observed'][:4].to(device)
fut_test = sample['future'][:4].to(device)

with torch.no_grad():
    pred_test = wrapper_test(obs_test)

print(f'Input shape  : {obs_test.shape}')
print(f'Output shape : {pred_test.shape}')
assert pred_test.shape == (4, 25, 17, 3)
print()

# Verify residuals are small
last_frame  = obs_test[:, -1:, :, :]
target_res  = fut_test - last_frame
pred_res    = wrapper_test.base_model(obs_test)

print(f'Absolute future range  : '
      f'[{fut_test.min():.3f}, {fut_test.max():.3f}]')
print(f'Residual target range  : '
      f'[{target_res.min():.3f}, {target_res.max():.3f}]')
print(f'Predicted resid range  : '
      f'[{pred_res.min():.3f}, {pred_res.max():.3f}]')
print()
print('Residuals are much smaller numbers than absolutes:')
print(f'  Absolute range width: '
      f'{fut_test.max().item()-fut_test.min().item():.3f}m')
print(f'  Residual range width: '
      f'{target_res.max().item()-target_res.min().item():.3f}m')
print()
print('✓ Wrapper verified — residuals are smaller targets')
print('✓ Output shape correct')
print('✓ Ready to train')

del wrapper_test, base_m1

Input shape  : torch.Size([4, 10, 17, 3])
Output shape : torch.Size([4, 25, 17, 3])

Absolute future range  : [-0.756, 1.705]
Residual target range  : [-0.610, 0.694]
Predicted resid range  : [-5.252, 4.123]

Residuals are much smaller numbers than absolutes:
  Absolute range width: 2.461m
  Residual range width: 1.304m

✓ Wrapper verified — residuals are smaller targets
✓ Output shape correct
✓ Ready to train


## 03. Train M1_VR: Vanilla Transformer with Velocity Residuals

Identical architecture to M1. Only the prediction target changes.
The model learns to predict motion deltas instead of absolute positions.

In [9]:
from models.vanilla_transformer import VanillaTransformer

# Build model and wrap it
base_M1 = VanillaTransformer(
    J=17, D=3, d_model=256, n_heads=4,
    n_enc_layers=2, n_dec_layers=2,
    d_ff=512, dropout=0.1, t_obs=10, t_pred=25
).to(device)

model_M1_VR = VelocityResidualWrapper(base_M1).to(device)

print(f'M1_VR parameters: {model_M1_VR.count_parameters():,}')
print('(Same as original M1 — wrapper adds no parameters)')

# Quick forward pass check
with torch.no_grad():
    _test = model_M1_VR(obs_test)
assert _test.shape == (4, 25, 17, 3)
print('✓ M1_VR forward pass correct')

M1_VR parameters: 2,668,595
(Same as original M1 — wrapper adds no parameters)
✓ M1_VR forward pass correct


In [10]:
trained_M1_VR = train_model_velres(
    model_wrapper   = model_M1_VR,
    train_loader    = train_loader,
    test_loader     = test_loader,
    n_epochs        = 50,
    lr              = 1e-3,
    device          = device,
    experiment_name = 'M1_VR_vanilla_transformer',
    eval_every      = 5,
    save_dir        = SAVE_DIR,
    log_dir         = LOG_DIR,
    resume          = False
)

Starting: M1_VR_vanilla_transformer
Epoch   5/50 | loss: 0.00851 | MPJPE@560ms: 103.5mm | GVR: 0.0007
  ✓ Best saved: MPJPE@560ms=103.5mm
Epoch  10/50 | loss: 0.00772 | MPJPE@560ms: 108.2mm | GVR: 0.0017
Epoch  15/50 | loss: 0.00730 | MPJPE@560ms: 107.7mm | GVR: 0.0008
Epoch  20/50 | loss: 0.00773 | MPJPE@560ms: 115.5mm | GVR: 0.0022
Epoch  25/50 | loss: 0.00766 | MPJPE@560ms: 120.1mm | GVR: 0.0004
Epoch  30/50 | loss: 0.00737 | MPJPE@560ms: 114.9mm | GVR: 0.0009
Epoch  35/50 | loss: 0.00642 | MPJPE@560ms: 106.6mm | GVR: 0.0003
Epoch  40/50 | loss: 0.00614 | MPJPE@560ms: 107.1mm | GVR: 0.0003
Epoch  45/50 | loss: 0.00600 | MPJPE@560ms: 105.5mm | GVR: 0.0034
Epoch  50/50 | loss: 0.00618 | MPJPE@560ms: 118.5mm | GVR: 0.0013

Done. Best MPJPE@560ms: 103.5mm
Best checkpoint: /content/drive/MyDrive/L4_Research_Resources/SP_GaRT_VelRes/checkpoints/M1_VR_vanilla_transformer_best.pth


## 04. Train M2_VR: Dense Graph Transformer with Velocity Residuals

Same graph structure, anatomical attention bias, spatio-temporal blocks.
Only the prediction target changes from absolute to residual.

In [11]:
import importlib
import models.graph_transformer as gt_module
importlib.reload(gt_module)
from models.graph_transformer import DenseGraphTransformer

base_M2 = DenseGraphTransformer(
    J=17, D=3, d_model=256, n_heads=4,
    n_st_layers=2, d_ff=512,
    dropout=0.1, t_obs=10, t_pred=25
).to(device)

model_M2_VR = VelocityResidualWrapper(base_M2).to(device)

print(f'M2_VR parameters: {model_M2_VR.count_parameters():,}')

with torch.no_grad():
    _test = model_M2_VR(obs_test)
assert _test.shape == (4, 25, 17, 3)
print('✓ M2_VR forward pass correct')

M2_VR parameters: 4,827,195
✓ M2_VR forward pass correct


In [12]:
trained_M2_VR = train_model_velres(
    model_wrapper   = model_M2_VR,
    train_loader    = train_loader,
    test_loader     = test_loader,
    n_epochs        = 50,
    lr              = 1e-4,
    device          = device,
    experiment_name = 'M2_VR_dense_graph_transformer',
    eval_every      = 5,
    save_dir        = SAVE_DIR,
    log_dir         = LOG_DIR,
    resume          = False
)

Starting: M2_VR_dense_graph_transformer
Epoch   5/50 | loss: 0.00965 | MPJPE@560ms: 115.9mm | GVR: 0.0058
  ✓ Best saved: MPJPE@560ms=115.9mm
Epoch  10/50 | loss: 0.00704 | MPJPE@560ms: 105.2mm | GVR: 0.0039
  ✓ Best saved: MPJPE@560ms=105.2mm
Epoch  15/50 | loss: 0.00594 | MPJPE@560ms: 96.6mm | GVR: 0.0005
  ✓ Best saved: MPJPE@560ms=96.6mm
Epoch  20/50 | loss: 0.00527 | MPJPE@560ms: 96.3mm | GVR: 0.0027
  ✓ Best saved: MPJPE@560ms=96.3mm
Epoch  25/50 | loss: 0.00472 | MPJPE@560ms: 97.0mm | GVR: 0.0021
Epoch  30/50 | loss: 0.00428 | MPJPE@560ms: 100.5mm | GVR: 0.0003
Epoch  35/50 | loss: 0.00365 | MPJPE@560ms: 98.3mm | GVR: 0.0013
Epoch  40/50 | loss: 0.00341 | MPJPE@560ms: 102.6mm | GVR: 0.0018
Epoch  45/50 | loss: 0.00322 | MPJPE@560ms: 103.8mm | GVR: 0.0012
Epoch  50/50 | loss: 0.00306 | MPJPE@560ms: 101.7mm | GVR: 0.0009

Done. Best MPJPE@560ms: 96.3mm
Best checkpoint: /content/drive/MyDrive/L4_Research_Resources/SP_GaRT_VelRes/checkpoints/M2_VR_dense_graph_transformer_best.pth


## 05. Train M3a_VR: Heuristic Pruned Transformer with Velocity Residuals

Gaze + proximity soft attention mask is completely preserved in the encoder.
The pruning mask is still computed from the observed skeleton every forward pass.

In [13]:
import importlib
import models.pruned_graph_transformer as pgt_module
importlib.reload(pgt_module)
from models.pruned_graph_transformer import PrunedGraphTransformer

base_M3a = PrunedGraphTransformer(
    J=17, D=3, d_model=256, n_heads=4,
    n_st_layers=2, d_ff=512,
    dropout=0.1, t_obs=10, t_pred=25
).to(device)

model_M3a_VR = VelocityResidualWrapper(base_M3a).to(device)

print(f'M3a_VR parameters: {model_M3a_VR.count_parameters():,}')

with torch.no_grad():
    _test = model_M3a_VR(obs_test)
assert _test.shape == (4, 25, 17, 3)
print('✓ M3a_VR forward pass correct')

M3a_VR parameters: 4,827,195
✓ M3a_VR forward pass correct


In [14]:
trained_M3a_VR = train_model_velres(
    model_wrapper   = model_M3a_VR,
    train_loader    = train_loader,
    test_loader     = test_loader,
    n_epochs        = 50,
    lr              = 1e-4,
    device          = device,
    experiment_name = 'M3a_VR_pruned_graph_transformer',
    eval_every      = 5,
    save_dir        = SAVE_DIR,
    log_dir         = LOG_DIR,
    resume          = False
)

Starting: M3a_VR_pruned_graph_transformer
Epoch   5/50 | loss: 0.00951 | MPJPE@560ms: 112.6mm | GVR: 0.0000
  ✓ Best saved: MPJPE@560ms=112.6mm
Epoch  10/50 | loss: 0.00707 | MPJPE@560ms: 101.8mm | GVR: 0.0007
  ✓ Best saved: MPJPE@560ms=101.8mm
Epoch  15/50 | loss: 0.00600 | MPJPE@560ms: 99.4mm | GVR: 0.0062
  ✓ Best saved: MPJPE@560ms=99.4mm
Epoch  20/50 | loss: 0.00531 | MPJPE@560ms: 99.4mm | GVR: 0.0008
  ✓ Best saved: MPJPE@560ms=99.4mm
Epoch  25/50 | loss: 0.00478 | MPJPE@560ms: 97.9mm | GVR: 0.0108
  ✓ Best saved: MPJPE@560ms=97.9mm
Epoch  30/50 | loss: 0.00430 | MPJPE@560ms: 97.5mm | GVR: 0.0019
  ✓ Best saved: MPJPE@560ms=97.5mm
Epoch  35/50 | loss: 0.00362 | MPJPE@560ms: 102.0mm | GVR: 0.0006
Epoch  40/50 | loss: 0.00340 | MPJPE@560ms: 100.0mm | GVR: 0.0003
Epoch  45/50 | loss: 0.00320 | MPJPE@560ms: 102.6mm | GVR: 0.0015
Epoch  50/50 | loss: 0.00303 | MPJPE@560ms: 102.7mm | GVR: 0.0032

Done. Best MPJPE@560ms: 97.5mm
Best checkpoint: /content/drive/MyDrive/L4_Research_Resour

## 06. Train M4_VR: Dense Graph + Gravity Loss + Velocity Residuals

The gravity-consistency loss is identical to the original M4.
It always checks ABSOLUTE joint positions (reconstructed before the check).
The physics constraint is completely unchanged.

In [15]:
base_M4 = DenseGraphTransformer(
    J=17, D=3, d_model=256, n_heads=4,
    n_st_layers=2, d_ff=512,
    dropout=0.1, t_obs=10, t_pred=25
).to(device)

model_M4_VR = VelocityResidualWrapper(base_M4).to(device)

print(f'M4_VR parameters: {model_M4_VR.count_parameters():,}')

with torch.no_grad():
    _test = model_M4_VR(obs_test)
assert _test.shape == (4, 25, 17, 3)
print('✓ M4_VR forward pass correct')

# Verify gravity loss still works on reconstructed absolute positions
loss_check = VelResGravityLoss(lambda_gravity=0.01)
last_f = obs_test[:, -1:, :, :]
pred_r = model_M4_VR.base_model(obs_test)
total_c, recon_c, grav_c = loss_check(pred_r, fut_test, obs_test, last_f)
print(f'Loss check — recon: {recon_c.item():.4f}, '
      f'gravity: {grav_c.item():.4f}, '
      f'gravity%: {grav_c.item()*0.01/total_c.item()*100:.1f}%')
print('✓ Gravity loss verified on absolute positions')

M4_VR parameters: 4,827,195
✓ M4_VR forward pass correct
Loss check — recon: 1.8920, gravity: 0.2262, gravity%: 0.1%
✓ Gravity loss verified on absolute positions


In [16]:
trained_M4_VR = train_model_velres_gravity(
    model_wrapper   = model_M4_VR,
    train_loader    = train_loader,
    test_loader     = test_loader,
    n_epochs        = 50,
    lr              = 1e-4,
    device          = device,
    experiment_name = 'M4_VR_SP_GaRT',
    lambda_gravity  = 0.01,
    eval_every      = 5,
    save_dir        = SAVE_DIR,
    log_dir         = LOG_DIR,
    resume          = False
)

Starting: M4_VR_SP_GaRT
Epoch   5/50 | loss: 0.00968 (recon: 0.00961, grav: 0.00670) | MPJPE@560ms: 118.0mm | GVR: 0.0001
  ✓ Best saved: MPJPE@560ms=118.0mm
Epoch  10/50 | loss: 0.00727 (recon: 0.00723, grav: 0.00395) | MPJPE@560ms: 104.6mm | GVR: 0.0001
  ✓ Best saved: MPJPE@560ms=104.6mm
Epoch  15/50 | loss: 0.00616 (recon: 0.00613, grav: 0.00348) | MPJPE@560ms: 96.2mm | GVR: 0.0032
  ✓ Best saved: MPJPE@560ms=96.2mm
Epoch  20/50 | loss: 0.00551 (recon: 0.00548, grav: 0.00292) | MPJPE@560ms: 100.1mm | GVR: 0.0005
Epoch  25/50 | loss: 0.00492 (recon: 0.00489, grav: 0.00256) | MPJPE@560ms: 108.5mm | GVR: 0.0008
Epoch  30/50 | loss: 0.00444 (recon: 0.00442, grav: 0.00233) | MPJPE@560ms: 98.1mm | GVR: 0.0001
Epoch  35/50 | loss: 0.00376 (recon: 0.00374, grav: 0.00212) | MPJPE@560ms: 97.5mm | GVR: 0.0008
Epoch  40/50 | loss: 0.00355 (recon: 0.00353, grav: 0.00198) | MPJPE@560ms: 104.2mm | GVR: 0.0009
Epoch  45/50 | loss: 0.00334 (recon: 0.00332, grav: 0.00191) | MPJPE@560ms: 97.5mm | GVR

## 07. Train M5_VR: True SP-GaRT (Pruned Graph + Gravity) + Velocity Residuals

Both novel contributions (pruning + gravity) combined with velocity residuals.
This is the complete SP-GaRT model with the improved prediction representation.

In [17]:
base_M5 = PrunedGraphTransformer(
    J=17, D=3, d_model=256, n_heads=4,
    n_st_layers=2, d_ff=512,
    dropout=0.1, t_obs=10, t_pred=25
).to(device)

model_M5_VR = VelocityResidualWrapper(base_M5).to(device)

print(f'M5_VR parameters: {model_M5_VR.count_parameters():,}')

with torch.no_grad():
    _test = model_M5_VR(obs_test)
assert _test.shape == (4, 25, 17, 3)
print('✓ M5_VR forward pass correct')

M5_VR parameters: 4,827,195
✓ M5_VR forward pass correct


In [18]:
trained_M5_VR = train_model_velres_gravity(
    model_wrapper   = model_M5_VR,
    train_loader    = train_loader,
    test_loader     = test_loader,
    n_epochs        = 50,
    lr              = 1e-4,
    device          = device,
    experiment_name = 'M5_VR_true_SP_GaRT',
    lambda_gravity  = 0.01,
    eval_every      = 5,
    save_dir        = SAVE_DIR,
    log_dir         = LOG_DIR,
    resume          = False
)

Starting: M5_VR_true_SP_GaRT
Epoch   5/50 | loss: 0.00983 (recon: 0.00976, grav: 0.00695) | MPJPE@560ms: 115.8mm | GVR: 0.0057
  ✓ Best saved: MPJPE@560ms=115.8mm
Epoch  10/50 | loss: 0.00732 (recon: 0.00728, grav: 0.00397) | MPJPE@560ms: 113.0mm | GVR: 0.0000
  ✓ Best saved: MPJPE@560ms=113.0mm
Epoch  15/50 | loss: 0.00616 (recon: 0.00612, grav: 0.00336) | MPJPE@560ms: 103.0mm | GVR: 0.0000
  ✓ Best saved: MPJPE@560ms=103.0mm
Epoch  20/50 | loss: 0.00546 (recon: 0.00543, grav: 0.00291) | MPJPE@560ms: 94.8mm | GVR: 0.0003
  ✓ Best saved: MPJPE@560ms=94.8mm
Epoch  25/50 | loss: 0.00492 (recon: 0.00490, grav: 0.00259) | MPJPE@560ms: 96.6mm | GVR: 0.0000
Epoch  30/50 | loss: 0.00449 (recon: 0.00446, grav: 0.00226) | MPJPE@560ms: 100.9mm | GVR: 0.0008
Epoch  35/50 | loss: 0.00378 (recon: 0.00376, grav: 0.00195) | MPJPE@560ms: 99.1mm | GVR: 0.0011
Epoch  40/50 | loss: 0.00355 (recon: 0.00353, grav: 0.00182) | MPJPE@560ms: 100.5mm | GVR: 0.0019
Epoch  45/50 | loss: 0.00334 (recon: 0.00332, g

## 08. Evaluate All VelRes Models

Load all trained VelRes checkpoints and compare with originals.

In [19]:
from utils.trainer import evaluate_model, print_results, measure_inference_time
from collections import OrderedDict

def load_velres_model(base_class, base_kwargs,
                       ckpt_path, device):
    """Load a VelocityResidualWrapper from checkpoint."""
    base  = base_class(**base_kwargs).to(device)
    model = VelocityResidualWrapper(base).to(device)
    ckpt  = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt['model_state'])
    model.eval()
    print(f'  ✓ {ckpt_path.split("/")[-1]} '
          f'ep={ckpt["epoch"]} '
          f'best={ckpt["best_mpjpe"]:.1f}mm')
    return model

GRAPH_KWARGS = dict(
    J=17, D=3, d_model=256, n_heads=4,
    n_st_layers=2, d_ff=512,
    dropout=0.1, t_obs=10, t_pred=25
)
M1_KWARGS = dict(
    J=17, D=3, d_model=256, n_heads=4,
    n_enc_layers=2, n_dec_layers=2,
    d_ff=512, dropout=0.1, t_obs=10, t_pred=25
)

print('Loading VelRes models...')
m1_vr = load_velres_model(
    VanillaTransformer, M1_KWARGS,
    f'{SAVE_DIR}/M1_VR_vanilla_transformer_best.pth', device
)
m2_vr = load_velres_model(
    DenseGraphTransformer, GRAPH_KWARGS,
    f'{SAVE_DIR}/M2_VR_dense_graph_transformer_best.pth', device
)
m3a_vr = load_velres_model(
    PrunedGraphTransformer, GRAPH_KWARGS,
    f'{SAVE_DIR}/M3a_VR_pruned_graph_transformer_best.pth', device
)
m4_vr = load_velres_model(
    DenseGraphTransformer, GRAPH_KWARGS,
    f'{SAVE_DIR}/M4_VR_SP_GaRT_best.pth', device
)
m5_vr = load_velres_model(
    PrunedGraphTransformer, GRAPH_KWARGS,
    f'{SAVE_DIR}/M5_VR_true_SP_GaRT_best.pth', device
)
print('\nAll VelRes models loaded.')

Loading VelRes models...
  ✓ M1_VR_vanilla_transformer_best.pth ep=5 best=103.5mm
  ✓ M2_VR_dense_graph_transformer_best.pth ep=20 best=96.3mm
  ✓ M3a_VR_pruned_graph_transformer_best.pth ep=30 best=97.5mm
  ✓ M4_VR_SP_GaRT_best.pth ep=15 best=96.2mm
  ✓ M5_VR_true_SP_GaRT_best.pth ep=20 best=94.8mm

All VelRes models loaded.


In [20]:
print('Evaluating on full test set...')

vr_results = {}
vr_speeds  = {}

for name, model in [('M1_VR', m1_vr), ('M2_VR', m2_vr),
                     ('M3a_VR', m3a_vr), ('M4_VR', m4_vr),
                     ('M5_VR', m5_vr)]:
    print(f'\n{name}:')
    res = evaluate_model(model, test_loader, device)
    ms, _  = measure_inference_time(model, device)
    vr_results[name] = res
    vr_speeds[name]  = ms
    print_results(res, name)
    print(f'Inference: {ms:.3f}ms')

Evaluating on full test set...

M1_VR:

  M1_VR
  MPJPE (mm) at horizons:
        80ms :    28.31 mm
       160ms :    44.35 mm
       320ms :    74.10 mm
       560ms :   115.32 mm
      1000ms :   189.66 mm
  ADE :   107.56 mm
  FDE :   189.66 mm
  GVR :   0.0325
  BLE :     9.72 mm
Inference: 2.538ms

M2_VR:

  M2_VR
  MPJPE (mm) at horizons:
        80ms :    22.83 mm
       160ms :    34.96 mm
       320ms :    60.29 mm
       560ms :    99.76 mm
      1000ms :   173.74 mm
  ADE :    94.06 mm
  FDE :   173.74 mm
  GVR :   0.0207
  BLE :    12.17 mm
Inference: 5.016ms

M3a_VR:

  M3a_VR
  MPJPE (mm) at horizons:
        80ms :    23.26 mm
       160ms :    35.23 mm
       320ms :    61.58 mm
       560ms :   100.68 mm
      1000ms :   172.90 mm
  ADE :    94.30 mm
  FDE :   172.90 mm
  GVR :   0.0196
  BLE :    12.19 mm
Inference: 5.190ms

M4_VR:

  M4_VR
  MPJPE (mm) at horizons:
        80ms :    25.14 mm
       160ms :    37.26 mm
       320ms :    63.19 mm
       560ms :   100.

In [21]:
# ── Comparison table: Original vs VelRes ─────────────────────
print('=' * 80)
print('COMPARISON: ORIGINAL vs VELOCITY RESIDUAL MODELS')
print('=' * 80)
print(f'{"Model":<22} {"80ms":>6} {"160ms":>6} '
      f'{"320ms":>6} {"560ms":>6} {"1000ms":>7} '
      f'{"GVR":>8} {"BLE":>7} {"ms":>6}')
print('-' * 80)

# Original results (previously confirmed numbers)
originals = {
    'M1 (original)':  (83.3, 89.1, 107.8, 142.4, 207.3, 0.0548, 23.37, 2.46),
    'M2 (original)':  (61.6, 67.1,  85.3, 119.1, 183.8, 0.0200, 24.14, 4.49),
    'M3a (original)': (58.9, 64.3,  82.5, 115.7, 179.7, 0.0504, 22.18, 4.98),
    'M4 (original)':  (58.1, 63.9,  82.6, 115.9, 180.7, 0.0073, 23.97, 4.21),
    'M5 (original)':  (63.7, 68.6,  85.8, 118.4, 182.9, 0.0074, 22.82, 4.83),
}

for name, vals in originals.items():
    h80,h160,h320,h560,h1000,gvr,ble,ms = vals
    print(f'{name:<22} {h80:>6.1f} {h160:>6.1f} '
          f'{h320:>6.1f} {h560:>6.1f} {h1000:>7.1f} '
          f'{gvr:>8.4f} {ble:>7.2f} {ms:>6.2f}')

print('-' * 80)
print('VELOCITY RESIDUAL VERSIONS:')
print('-' * 80)

for name, res in vr_results.items():
    gvr = res['gvr'] or 0
    ms  = vr_speeds[name]
    print(f'{name:<22} '
          f'{res["mpjpe"][80]:>6.1f} '
          f'{res["mpjpe"][160]:>6.1f} '
          f'{res["mpjpe"][320]:>6.1f} '
          f'{res["mpjpe"][560]:>6.1f} '
          f'{res["mpjpe"][1000]:>7.1f} '
          f'{gvr:>8.4f} '
          f'{res["ble"]:>7.2f} '
          f'{ms:>6.2f}')

print('=' * 80)
print()
print('IMPROVEMENT FROM VELOCITY RESIDUALS:')
orig_560 = {'M1': 142.4, 'M2': 119.1, 'M3a': 115.7,
             'M4': 115.9, 'M5': 118.4}
for vr_name, res in vr_results.items():
    orig_name = vr_name.replace('_VR', '')
    if orig_name in orig_560:
        orig = orig_560[orig_name]
        new  = res['mpjpe'][560]
        diff = orig - new
        pct  = diff / orig * 100
        direction = '✓ better' if diff > 0 else '✗ worse'
        print(f'  {vr_name}: {orig:.1f} → {new:.1f}mm '
              f'({diff:+.1f}mm, {pct:+.1f}%) {direction}')

COMPARISON: ORIGINAL vs VELOCITY RESIDUAL MODELS
Model                    80ms  160ms  320ms  560ms  1000ms      GVR     BLE     ms
--------------------------------------------------------------------------------
M1 (original)            83.3   89.1  107.8  142.4   207.3   0.0548   23.37   2.46
M2 (original)            61.6   67.1   85.3  119.1   183.8   0.0200   24.14   4.49
M3a (original)           58.9   64.3   82.5  115.7   179.7   0.0504   22.18   4.98
M4 (original)            58.1   63.9   82.6  115.9   180.7   0.0073   23.97   4.21
M5 (original)            63.7   68.6   85.8  118.4   182.9   0.0074   22.82   4.83
--------------------------------------------------------------------------------
VELOCITY RESIDUAL VERSIONS:
--------------------------------------------------------------------------------
M1_VR                    28.3   44.4   74.1  115.3   189.7   0.0325    9.72   2.54
M2_VR                    22.8   35.0   60.3   99.8   173.7   0.0207   12.17   5.02
M3a_VR          